# Temas principais por candidato

Monta o top 10 de temas de cada perfil monitorado, com o número de posts atrás de cada tema, e um resumo de duas ou três frases factuais sobre o que a conta vem publicando.

O notebook usa as funções de `classificar.py`. Suba os dois arquivos juntos, mais o `credentials.json`.

## Por que a contagem não sai do modelo

A coluna `Temas` da aba **Alinhamento Temático** é o resultado do `tematica.py` já deduplicado. Em 702 temas do ACM Neto há 702 strings distintas: zero repetição. Um tema que apareceu em 30 posts e outro que apareceu em 1 chegam ao modelo como duas linhas de peso igual.

Sem peso não existe top 5. O que sai dali é uma lista sem ordem, em que `Cultura` (4 posts de 126) fica lado a lado com `Saúde` (dezenas).

Aqui a unidade é o post. O Python conta, o modelo só agrupa as variações de escrita do mesmo assunto.


In [ ]:
!pip install -q gspread google-genai google-auth


## 1. Conexão

O `credentials.json` é a conta de serviço do Google com a chave do Gemini dentro, no campo `GEMINI_API_KEY`. Precisa estar na mesma pasta.


In [ ]:
import classificar as C
from google import genai

sh = C.gs_client().open_by_key(C.SPREADSHEET_ID)
gem = genai.Client(api_key=C.carregar_gemini_api_key())
print('abas:', [w.title for w in sh.worksheets()])


## 2. Ler os posts

Lê as abas mensais, não a aba de alinhamento. Cada post traz sua própria lista de temas, e é essa lista que dá a frequência.

Duas coisas são tratadas aqui:

- Em 142 posts de agosto o rotulador devolveu tudo numa linha só separado por vírgula, em vez de bullets. Sem tratar, esses posts contam como um tema gigante e o que eles tinham some do ranking.
- Post marcado como `(mídia expirada)` ou `(post fora do ar)` não é conteúdo e fica de fora da conta.


In [ ]:
posts = C.ler_posts(sh)
ufs = C.ler_uf(sh)

print(f'{sum(len(v) for v in posts.values())} posts de {len(posts)} candidatos')
print()
for cand, lista in sorted(posts.items(), key=lambda kv: -len(kv[1]))[:5]:
    print(f'  {len(lista):>4}  {cand}')
print('  ...')
for cand, lista in sorted(posts.items(), key=lambda kv: len(kv[1]))[:5]:
    print(f'  {len(lista):>4}  {cand}')


## 3. Contar

Conta em quantos **posts** o tema aparece, não quantas vezes ele é escrito. Um post que repete `Saúde` em três bullets vale um post, não três.

Antes de contar, dois filtros rodam no Python:

- **Lista `RUIDO`**: processo de campanha (comício, adesivaço, convenção) e emoção solta (gratidão, esperança, orgulho). Fica no código, e não no prompt, porque assim dá para discutir item a item com o cliente e o resultado é o mesmo em toda rodada. A comparação é por igualdade exata, nunca por `contém`: senão `campanha eleitoral` derrubaria junto `financiamento de campanha`.
- **Nome do candidato e do estado**: `Bahia` aparecia em 50 dos 126 posts do ACM Neto e `Pernambuco` em 68 dos 137 da Raquel Lyra. Lideram a contagem e não dizem nada sobre pauta.


In [ ]:
CANDIDATO = 'ACM Neto'

ruido = C.ruido_do_candidato(CANDIDATO, ufs.get(CANDIDATO, ''))
contagem, canonico, analisados, posts_por_tema = C.contar(posts[CANDIDATO], ruido)

print(f'{CANDIDATO}: {len(posts[CANDIDATO])} posts, {analisados} com temas, {len(contagem)} temas distintos')
print()
for chave, n in contagem.most_common(20):
    print(f'  {n:>3} posts  {canonico[chave]}')


### O que o filtro derrubou

Vale olhar de vez em quando. Se algo que interessa estiver caindo aqui, é a lista `RUIDO` que precisa mudar.


In [ ]:
cortados, _, _, _ = C.contar(posts[CANDIDATO])   # sem o ruído do candidato
com_filtro = set(contagem)

print('temas cortados que apareceriam no topo:')
for chave, n in cortados.most_common(40):
    if chave not in com_filtro and n >= 3:
        print(f'  {n:>3} posts  {chave}')


## 4. A chamada ao modelo

O modelo recebe a lista de temas **com a contagem ao lado** e devolve JSON: os grupos e, dentro de cada grupo, exatamente quais temas brutos entraram.

Duas escolhas que importam:

- **`response_schema`**: a resposta vem como JSON validado, não como texto para parsear. Parseando texto solto, um `Aqui estão os temas:` no começo da resposta vira um tema.
- **`thinking_budget=0`**: agrupar string não precisa de raciocínio, e o pensamento é cobrado como saída sem aparecer no `candidates_token_count`. É o que faz a estimativa de custo bater com a fatura.

A regra do rótulo é o que separa útil de genérico: se os temas brutos trazem nome de programa ou de indicador, o rótulo leva junto. `Saúde: fila de regulação e PIX Saúde` diz o que o candidato fala. `Saúde` não diz.


In [ ]:
grupos, resumo_bruto, uso = C.classificar(gem, CANDIDATO, contagem, canonico, posts_por_tema)

for g in grupos:
    print(g['rotulo'])
    print('   ', ', '.join(g['brutos'][:6]))
print()
print('resumo do modelo:', resumo_bruto)
print('tokens:', uso)


## 5. Guardas antes de gravar

`consolidar` calcula a união dos posts no Python e joga fora todo tema bruto que o modelo devolveu mas que não estava na entrada. Se um post traz `Saúde` e `Saúde Pública`, ele conta uma vez no grupo. Grupo que fica sem nenhum bruto válido some inteiro.

`montar_resumo` faz o mesmo com o texto:

- A frase com os números é escrita pelo Python, com o que foi contado. O modelo é proibido de escrever número, e frase com dígito é cortada.
- Nome próprio citado tem que existir na lista que o modelo recebeu. Se não existir, a frase sai.

Mesma checagem que o `analise_planos.py` faz antes de gravar citação.


In [ ]:
temas = C.consolidar(grupos, contagem, canonico, analisados, posts_por_tema)
resumo = C.montar_resumo(CANDIDATO, resumo_bruto, temas, analisados,
                         len(posts[CANDIDATO]), contagem, canonico)

for i, t in enumerate(temas, 1):
    print(f"{i}. {t['rotulo']} - {t['posts']} posts ({t['pct']}%)")
    print(f"     {', '.join(t['brutos'])}")
print()
print(resumo)


## 6. Rodar tudo e gravar

Grava cinco colunas na aba **Alinhamento Temático**: `Temas Principais`, `Nº de posts`, `% dos posts em que aparece`, `Temas brutos agrupados` e `Resumo da conta`. Um post pode ter mais de um tema, então os percentuais não devem ser somados.

Detalhes do comportamento:

- Escreve de 10 em 10 candidatos. Se cair no meio, o que já foi feito está gravado.
- Pula quem já tem `Temas Principais` preenchido. Use `--forcar` para refazer.
- Candidato com menos de 20 posts analisados não é ranqueado: sai marcado como base pequena. Um top 10 sobre 13 posts, que é o caso do Adailton Fúria, não descreve nada.

Rodada completa dos 78: entre 10 e 15 minutos, custo abaixo de US$ 0,50.


In [ ]:
!python classificar.py --limite 3     # piloto
# !python classificar.py               # rodada completa
# !python classificar.py --forcar      # refaz todo mundo


## Não precisa rodar nada antes

O script se vira sozinho. A lista de candidatos sai da aba **Instagram**, que é o cadastro do monitoramento, filtrada por quem tem post coletado. Se a aba **Alinhamento Temático** não existir, ele cria; se faltar candidato nela, ele acrescenta a linha.

Usar o cadastro como fonte também tira `PL CE` e `Solidariedade`, que aparecem na coluna Candidato das abas mensais como se fossem candidatos.

Uma ressalva: se você rodar o `tematica.py` **depois**, ele reescreve a aba inteira e leva junto as cinco colunas. O `classificar.py` não depende mais dele, então o mais simples é não rodar os dois na mesma aba.
